# Ensembles: combinar árvores para errar menos

**Para quem está começando.** Este notebook supõe que você já viu a árvore de decisão, na pasta `classical`. A ideia é rodar de cima para baixo, célula por célula, e olhar os números.

O que vamos fazer, em ordem:

1. entender bagging e boosting, as duas formas de combinar árvores;
2. rodar Random Forest, XGBoost, LightGBM e CatBoost na mesma divisão;
3. ver o que acontece quando o número de árvores cresce;
4. entender por que essa família domina dado em tabela.

Roda no Google Colab. A célula abaixo instala os três pacotes de boosting, que não vêm todos no Colab; a instalação leva um ou dois minutos.


## As duas formas de combinar árvores

Uma árvore sozinha decora o treino: ela é fácil de ajustar e fácil de exagerar, e foi o que o notebook de classificação mostrou. Combinar árvores resolve isso de duas maneiras diferentes, e vale separar bem.

**Bagging** é o caminho do Random Forest. Ele treina muitas árvores **independentes**, cada uma sorteando linhas (com reposição) e colunas do treino, e no fim vota a maioria. As árvores são profundas e diferentes entre si de propósito: o erro delas é parcialmente independente, e a média cancela parte do ruído. O que o bagging ataca é a **variância**.

**Boosting** é o caminho do XGBoost, do LightGBM e do CatBoost. Ele treina árvores **em sequência**, cada uma tentando corrigir o erro que sobrou das anteriores, e por isso cada árvore é rasa de propósito. O que o boosting ataca é o **viés**.

Três consequências que mudam o uso no dia a dia:

- o bagging roda em paralelo, porque as árvores não conversam entre si; o boosting é sequencial, e a árvore 50 depende da 49;
- o boosting tem mais botões (taxa de aprendizado, número de árvores, profundidade) e é onde as bibliotecas se diferenciam: o LightGBM cresce a árvore folha por folha, o CatBoost cuida de variável categórica e do viés de treino, o XGBoost regulariza o passo;
- nenhum dos dois precisa de padronização, pela mesma razão da árvore do notebook de classificação: eles comparam limiar dentro de cada coluna.


## Passo 0: ambiente e configuração

A semente é a mesma em todas as células. A divisão de treino e teste também é a do resto do repositório, para os números serem comparáveis com os outros notebooks.


In [ ]:
%pip install -q xgboost==3.4.1 lightgbm==4.7.0 catboost==1.2.10

In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
N_ARVORES = 150
plt.rcParams["figure.figsize"] = (9, 4)

data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.25, stratify=data.target, random_state=RANDOM_STATE,
)
print(f"✅ {X_train.shape[0]} linhas de treino | {X_test.shape[0]} de teste | {X_train.shape[1]} atributos")
print(f"   xgboost {xgb.__version__} | lightgbm {lgb.__version__}")

## Passo 1: a mesma régua para os cinco modelos

A árvore sozinha entra como referência do que acontece sem combinar. Os cinco são avaliados com validação cruzada de 5 folds no treino, e a acurácia de teste entra ao lado, como conferência.

Um detalhe de ajuste que não aparece nas métricas: cada biblioteca de boosting tem um conjunto de hiperparâmetros padrão razoável, e é isso que estamos usando. Quem for comparar de verdade precisa dar a mesma paciência de busca para todos, e é o que o próximo notebook da pasta faz.


In [ ]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

modelos = {
    "Árvore (referência)": DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=N_ARVORES, random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost": xgb.XGBClassifier(n_estimators=N_ARVORES, learning_rate=0.1, max_depth=4,
                                 random_state=RANDOM_STATE),
    "LightGBM": lgb.LGBMClassifier(n_estimators=N_ARVORES, learning_rate=0.1, max_depth=4,
                                   random_state=RANDOM_STATE, verbose=-1),
    # allow_writing_files=False: por padrao o CatBoost cria uma pasta catboost_info/ com o log
    "CatBoost": CatBoostClassifier(iterations=N_ARVORES, learning_rate=0.1, depth=4,
                                   random_state=RANDOM_STATE, verbose=0,
                                   allow_writing_files=False),
}


def timeit(fn, *args, **kwargs):
    """Run fn once and return (result, elapsed_seconds)."""
    inicio = time.perf_counter()
    resultado = fn(*args, **kwargs)
    return resultado, time.perf_counter() - inicio


linhas = []
for nome, modelo in modelos.items():
    _, fit_s = timeit(modelo.fit, X_train, y_train)
    _, predict_s = timeit(modelo.predict, X_test)
    acuracia = cross_val_score(modelo, X_train, y_train, cv=folds, scoring="accuracy")
    auc = cross_val_score(modelo, X_train, y_train, cv=folds, scoring="roc_auc")
    linhas.append({"modelo": nome, "acuracia_cv": acuracia.mean(), "desvio": acuracia.std(),
                   "roc_auc_cv": auc.mean(), "acuracia_teste": modelo.score(X_test, y_test),
                   "fit_s": fit_s, "predict_s": predict_s})

tabela = pd.DataFrame(linhas).set_index("modelo").sort_values("roc_auc_cv", ascending=False)
display(tabela.round(4))

### Como ler isso

As três bibliotecas de boosting ficam à frente da árvore sozinha, e o Random Forest fica no meio. É o resultado esperado nesta tabela pequena e bem separada, e vale ler com calma:

- compare sempre `desvio` antes de declarar vencedor. Se a diferença entre dois modelos é menor que o desvio deles entre folds, a diferença é da partição, não do modelo;
- `fit_s` é um instantâneo de uma medição, e essa coluna muda entre execuções: o primeiro modelo a treinar paga a inicialização das threads da biblioteca, e o mesmo modelo pode aparecer dez vezes mais lento só por causa disso. O que ela mostra de estável é que treinar 150 árvores custa mais que treinar uma, e que as implementações diferem muito entre si. Em tabela grande é essa coluna que decide, e por isso ela merece medição repetida, não uma passada só;
- `predict_s` é onde o Random Forest paga a conta das árvores profundas: aqui ele foi a previsão mais cara, e por uma ordem de grandeza.

E o que **não** mudou em relação ao notebook de classificação: nenhum dos cinco precisa de padronização. Você pode alimentar as colunas com as unidades originais, como está sendo feito aqui.


## Passo 2: o que o número de árvores faz

Agora um botão só, medido nos dois lados da família. O gráfico mostra a acurácia de validação cruzada conforme o número de árvores cresce, com tudo o mais fixo.


In [ ]:
valores = [1, 5, 10, 25, 50, 100, 200]
linhas = []
for n in valores:
    for nome, modelo in {
        "Random Forest": RandomForestClassifier(n_estimators=n, random_state=RANDOM_STATE, n_jobs=-1),
        "XGBoost": xgb.XGBClassifier(n_estimators=n, learning_rate=0.3, max_depth=4,
                                     random_state=RANDOM_STATE),
    }.items():
        acuracia = cross_val_score(modelo, X_train, y_train, cv=folds, scoring="accuracy")
        linhas.append({"arvores": n, "modelo": nome, "acuracia_cv": acuracia.mean(),
                       "desvio": acuracia.std()})

tabela_arvores = pd.DataFrame(linhas).pivot(index="arvores", columns="modelo", values="acuracia_cv")
display(tabela_arvores.round(4))

fig, ax = plt.subplots()
for coluna in tabela_arvores.columns:
    ax.plot(tabela_arvores.index, tabela_arvores[coluna], marker="o", label=coluna)
ax.set_xscale("log")
ax.set_xlabel("número de árvores")
ax.set_ylabel("acurácia (validação cruzada)")
ax.set_title("Mais árvores nem sempre é melhor, mas também não costuma piorar")
ax.legend()
plt.show()

### Como ler isso

No Random Forest a curva sobe rápido e depois oscila em volta de um patamar. Essa oscilação não é aprendizado: em teoria, árvore a mais nunca piora o voto, ela só reduz a variância. O que se vê depois do platô é o ruído da validação cruzada, e é por isso que a curva não é perfeitamente monótona na prática. O custo, esse sim, cresce de forma linear no tempo.

No boosting a história é outra, porque as árvores são sequenciais e cada uma mexe no que a anterior deixou. Aqui a curva sobe mais devagar e continua subindo até o fim da grade; com taxa de aprendizado alta e orçamento maior, é essa curva que pode virar para baixo quando o modelo passa a ajustar o ruído do treino. É por isso que boosting se usa com taxa baixa, muitas árvores e parada antecipada, que é o assunto do próximo notebook.

Com uma árvore só, os dois empatam, e é o esperado: sem combinação, boosting e bagging degeneram na mesma árvore.

## Resumo

Combinar árvores ataca o defeito da árvore sozinha, que é decorar o treino, e o faz de duas maneiras. O bagging treina árvores independentes em amostras do dado e reduz a variância pelo voto. O boosting treina árvores em sequência, cada uma corrigindo o erro das anteriores, e reduz o viés, com árvores rasas de propósito.

A escolha prática: boosting costuma ganhar a acurácia em tabela, ao custo de tempo de treino e de mais botões; Random Forest é mais difícil de errar e paraleliza de graça. Os dois dispensam padronização, ao contrário do kNN e do SVM.

**Próximo passo:** os mesmos botões do boosting, um por um, com o que cada um faz medido em vez de repetido.

### Referências

**Bagging**

Breiman, L. (2001). *Random forests.* Machine Learning 45(1), 5-32.

**Boosting**

Friedman, J. H. (2001). *Greedy function approximation: a gradient boosting machine.* The Annals of Statistics 29(5), 1189-1232.

Chen, T., & Guestrin, C. (2016). *XGBoost: a scalable tree boosting system.* Proceedings of KDD.

Ke, G., Meng, Q., Finley, T., Wang, T., Chen, W., Ma, W., Ye, Q., & Liu, T.-Y. (2017). *LightGBM: a highly efficient gradient boosting decision tree.* Advances in Neural Information Processing Systems (NeurIPS).

Prokhorenkova, L., Gusev, G., Vorobev, A., Dorogush, A. V., & Gulin, A. (2018). *CatBoost: unbiased boosting with categorical features.* Advances in Neural Information Processing Systems (NeurIPS).

```bibtex
@article{breiman2001random,
  title   = {Random forests},
  author  = {Breiman, Leo},
  journal = {Machine Learning},
  volume  = {45},
  number  = {1},
  pages   = {5--32},
  year    = {2001}
}

@article{friedman2001greedy,
  title   = {Greedy function approximation: a gradient boosting machine},
  author  = {Friedman, Jerome H.},
  journal = {The Annals of Statistics},
  volume  = {29},
  number  = {5},
  pages   = {1189--1232},
  year    = {2001}
}

@inproceedings{chen2016xgboost,
  title     = {XGBoost: a scalable tree boosting system},
  author    = {Chen, Tianqi and Guestrin, Carlos},
  booktitle = {Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining},
  year      = {2016}
}

@inproceedings{ke2017lightgbm,
  title     = {LightGBM: a highly efficient gradient boosting decision tree},
  author    = {Ke, Guolin and Meng, Qi and Finley, Thomas and Wang, Taifeng and Chen, Wei and
               Ma, Weidong and Ye, Qiwei and Liu, Tie-Yan},
  booktitle = {Advances in Neural Information Processing Systems (NeurIPS)},
  year      = {2017}
}

@inproceedings{prokhorenkova2018catboost,
  title     = {CatBoost: unbiased boosting with categorical features},
  author    = {Prokhorenkova, Liudmila and Gusev, Gleb and Vorobev, Aleksandr and
               Dorogush, Anna Veronika and Gulin, Andrey},
  booktitle = {Advances in Neural Information Processing Systems (NeurIPS)},
  year      = {2018}
}
```
